## Publish a Downhole Collection object

This example shows how to convert drilling data in CSV format into an Evo geoscience object using the Evo Python SDK.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

The following parameters must be provided:

- The client ID of your Evo application.
- The callback/redirect URL of your Evo application.

To obtain these app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

In [1]:
from evo.notebooks import ServiceManagerWidget

input_path = "sample-data"

# Evo app credentials
client_id = "daves-evo-client"  # Replace with your client ID
# redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id,
    # redirect_url=redirect_url,
).login()

/Users/david.knight/Developer/evo-python-sdk-fork/packages/evo-sdk-common/src/evo/notebooks/authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

### Use the Evo Python SDK to create an object client and a data client

In [2]:
from evo.objects import ObjectAPIClient

# The object client will manage your auth token and Geoscience Object API requests.
object_client = ObjectAPIClient(manager.get_environment(), manager.get_connector())

# The data client will manage saving your data as Parquet and publishing your data to Evo storage.
data_client = object_client.get_data_client(manager.cache)

### Define helper functions

These functions assist with assembling the elements and components of geoscience objects and for viewing the new object in the Evo portal.

In [3]:
import numpy as np
import pandas as pd
from IPython.display import HTML, display


def create_hole_id_mapping(hole_id_table, value_list):
    """
    Create a hole ID mapping table based on the hole ID table and the value list.

    Args:
        hole_id_table (pd.DataFrame): The hole ID lookup table.
        value_list (pd.DataFrame): The value list to create the mapping from.

    Returns:
        mapping_df (pd.DataFrame): The hole ID mapping table.
    """

    num_keys = len(hole_id_table.index)

    mapping_df = pd.DataFrame(list())
    mapping_df["hole_index"] = hole_id_table["key"]
    mapping_df["offset"] = [0] * num_keys
    mapping_df["count"] = [0] * num_keys

    mapping_df["hole_index"] = mapping_df["hole_index"].astype("int32")
    mapping_df["offset"] = mapping_df["offset"].astype("uint64")
    mapping_df["count"] = mapping_df["count"].astype("uint64")

    prev_value = ""
    key = ""
    count = 0
    offset = 0

    for index, row in value_list.iterrows():
        new_value = row["data"]

        if new_value != prev_value:
            if prev_value != "":
                mapping_df.loc[mapping_df["hole_index"] == key, "count"] = count
                mapping_df.loc[mapping_df["hole_index"] == key, "offset"] = offset
                offset += count

            mask = hole_id_table["value"] == new_value
            masked_df = hole_id_table[mask]
            try:
                key_row = masked_df.iloc[[0]]
            except IndexError:
                print("Ignoring this hole ID")
                continue

            key = key_row["key"].iloc[0]
            count = 1
            prev_value = new_value
        else:
            count += 1

    mapping_df.loc[mapping_df["hole_index"] == key, "count"] = count
    mapping_df.loc[mapping_df["hole_index"] == key, "offset"] = offset

    return mapping_df


def create_category_lookup_and_values(attribute):
    """
    Create a category lookup table and the associated column of mapped key values.

    Args:
        attribute (pd.DataFrame): An attribute of a geoscience object.

    Returns:
        table_df (pd.DataFrame): The category lookup table.
        values_df (pd.DataFrame): The associated column with mapped key values.
    """

    # Replace NaN with empty string
    attribute.replace(np.nan, "", regex=True, inplace=True)
    set_obj = set(attribute["data"])
    list_obj = list(set_obj)
    list_obj.sort()
    num_unique_elements = len(list_obj)

    # Create lookup table
    table_df = pd.DataFrame([])
    table_df["key"] = list(range(1, num_unique_elements + 1))
    table_df["value"] = list_obj

    # Create data column
    values_df = pd.DataFrame([])
    values_df["data"] = attribute["data"].map(table_df.set_index("value")["key"])
    return table_df, values_df


def build_portal_url(object_metadata):
    """
    Build the URL to view the object in the Evo Portal.

    Args:
        object_metadata (str): The object metadata object returned after the object is created.

    Returns:
        str: The URL to view the object in the Evo Portal.
    """

    hub_url = object_metadata.environment.hub_url
    hub_name = hub_url.split("://")[1].split(".")[0]
    org_id = object_metadata.environment.org_id
    workspace_id = object_metadata.environment.workspace_id
    object_id = object_metadata.id

    url = f"https://evo.seequent.com/{org_id}/workspaces/{hub_name}/{workspace_id}/viewer?id={object_id}"

    display(HTML(f'<a href="{url}" target="_blank">View object in the Evo Portal</a>'))

### Define object metadata

Geoscience object data must conform to a specific object schema. The `evo-schemas` package provides Pydantic models that make it easy to work with the equivalent JSON schemas. 
For this example we'll use v1.2.0 of the downhole-collection schema, via the relevant Pydantic model.

Enter values for these parameters that are required by the object schema.
- `object_hole_id`: The column name that represents your hole ID. This value should be the same across all input files.
- `object_name`: The name of the object.
- `object_path`: The file path where the object will be found.
- `object_epsg_code`: (Optional) The EPSG region code that matches the location of your data. Leave as `None` if not required.
- `object_tags`: (Optional) A dictionary of additional tags to be assigned to the object. Leave as `None` is not required.

In [4]:
import os
import uuid

from evo_schemas.components import Crs_V1_0_1_EpsgCode

# Set the name of the hole ID parameter in the data.
object_hole_id = "BHID"

# Set other object properties.
# Add a short random suffix so each run creates a unique object name.
random_suffix = uuid.uuid4().hex[:6]
object_name = f"DHC_SDK_demo_{random_suffix}"
if os.environ.get("CI"):
    from datetime import datetime, timezone

    object_name += f"_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}"
object_path = "Jupyter_Example"
object_epsg_code = 32650
object_tags = {"Source": "Jupyter Notebook", "Evo SDK": "0.1.5"}

# Define a coordinate reference system (CRS) for the object.
coordinate_reference_system = Crs_V1_0_1_EpsgCode(epsg_code=object_epsg_code)

# Create an empty list to store geoscience object collections.
collections = []

# Define the object path.
full_obj_path = f"{object_path}/{object_name}.json"

### Define object attributes and keys

In [5]:
# List all of the attributes to be included in the object. Every attribute must have a unique key associated with it.
# Keys must be unique across the entire object, and we recommend saving a reference to the keys for later use.
collar_attributes_ref = {
    "Hole Type": str(uuid.uuid4()),
    "Random Number": str(uuid.uuid4()),
}

survey_attributes_ref = {
    "Random Numeric": str(uuid.uuid4()),
}

object_attributes_ref = {
    "Wolfpass_WP_assay": {
        "CU_pct": str(uuid.uuid4()),
        "AU_gpt": str(uuid.uuid4()),
        "DENSITY": str(uuid.uuid4()),
    },
    "Wolfpass_WP_lith": {
        "ROCK": str(uuid.uuid4()),
        "grouped_lith": str(uuid.uuid4()),
        "Split_Dykes": str(uuid.uuid4()),
    },
}

depth_attribute_ref = {
    "Wolfpass_depths": {
        "ATTR": str(uuid.uuid4()),
    }
}

### Collar

In [6]:
input_path = "sample-data"

# Define input and output file paths.
input_file_path = f"{input_path}/Wolfpass_collar.csv"

# Load the collar file, count the number of hole IDs and sort the data based on the hole ID.
input_df = pd.read_csv(input_file_path)
num_hole_ids = len(input_df.index)
sorted_collar_df = input_df.sort_values([object_hole_id]).reset_index(drop=True)

sorted_collar_df.head()

,BHID,XCOLLAR,YCOLLAR,ZCOLLAR,maxdepth,Hole Type,Random Number
0,WP001,445198.218,494110.593,3057.757536,198.050003,DD,0.508293
1,WP002,445196.625,494109.656,3054.265408,222.100006,DD,0.646661
2,WP003,445260.563,493860.719,3009.038328,208.199997,DD,0.923033
3,WP004,445203.344,493720.656,3033.777664,200.000000,DD,0.929552
4,WP005,445291.563,493566.219,2957.434730,200.000000,DD,0.456014


### Hole ID table

Most components in our downhole collection object will reference the hole IDs defined in the collar table.

This means that we must create a 2-column dataframe that maps a unique `key` to a `hole ID`.

The `location` component of the downhole collection object makes use of this mapping, so we provide a 1-column dataframe that lists the keys in the order they are displayed in the input file.

In [7]:
import pandas as pd
from evo_schemas.elements import LookupTable_V1_0_1

# Create a dataframe for the hole IDs.
hole_id_table_df = pd.DataFrame()
hole_id_table_df["key"] = [i for i in range(1, num_hole_ids + 1)]
hole_id_table_df["value"] = sorted_collar_df[object_hole_id]
hole_id_table_df["key"] = hole_id_table_df["key"].astype("int32")
hole_id_table_component = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(hole_id_table_df))

### Hole ID values

In [8]:
import pandas as pd
from evo_schemas.components import CategoryData_V1_0_1
from evo_schemas.elements import IntegerArray1_V1_0_1

# Create a dataframe and generate a list from 1 to `n`, where `n` is the number of hole IDs.
# This table represents the list of hole IDs created in the previous step.
hole_id_values_df = pd.DataFrame()
hole_id_values_df["data"] = [i for i in range(1, num_hole_ids + 1)]
hole_id_values_df["data"] = hole_id_values_df["data"].astype("int32")
hole_id_values_component = IntegerArray1_V1_0_1.from_dict(data_client.save_dataframe(hole_id_values_df))

hole_id_component = CategoryData_V1_0_1(
    table=hole_id_table_component,
    values=hole_id_values_component,
)

### Coordinates

In [9]:
import pandas as pd
from evo_schemas.components import BoundingBox_V1_0_1
from evo_schemas.elements import FloatArray3_V1_0_1

# Create a dataframe and copy the required columns.
# NOTE: The columns must be renamed to `x`, `y` and `z`.
coordinates_df = pd.DataFrame()
coordinates_df[["x", "y", "z"]] = input_df[["XCOLLAR", "YCOLLAR", "ZCOLLAR"]]

# Create a `BoundingBox_V1_0_1` component for the bounding box.
bounding_box = BoundingBox_V1_0_1(
    min_x=coordinates_df["x"].min(),
    max_x=coordinates_df["x"].max(),
    min_y=coordinates_df["y"].min(),
    max_y=coordinates_df["y"].max(),
    min_z=coordinates_df["z"].min(),
    max_z=coordinates_df["z"].max(),
)

# Create a `FloatArray3_V1_0_1` element for the coordinates.
coordinates_component = FloatArray3_V1_0_1.from_dict(data_client.save_dataframe(coordinates_df))

In [10]:
import pandas as pd
from evo_schemas.components import (
    CategoryAttribute_V1_1_0,
    ContinuousAttribute_V1_1_0,
    NanCategorical_V1_0_1,
    NanContinuous_V1_0_1,
)
from evo_schemas.elements import FloatArray1_V1_0_1, IntegerArray1_V1_0_1, LookupTable_V1_0_1

# Load the collar attributes

collar_attributes = []

for heading_name, heading_key in collar_attributes_ref.items():
    print(heading_name)
    if heading_name == "Random Number":
        # Construct a FloatArray1 component.
        values_df = pd.DataFrame()
        values_df = input_df.loc[:, [heading_name]].copy().astype(float).reset_index(drop=True)
        values = FloatArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))

        attribute = ContinuousAttribute_V1_1_0(
            name=heading_name, key=heading_key, nan_description=NanContinuous_V1_0_1(values=[]), values=values
        )

        collar_attributes.append(attribute)
    elif heading_name == "Hole Type":
        # Construct a LookupTable and IntegerArray1 component.
        data_df = input_df.loc[:, [heading_name]].copy().reset_index(drop=True)
        data_df.rename(columns={heading_name: "data"}, inplace=True)
        table_df, values_df = create_category_lookup_and_values(data_df)
        table = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(table_df))
        values = IntegerArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df.astype("int32")))

        attribute = CategoryAttribute_V1_1_0(
            name=heading_name,
            key=heading_key,
            nan_description=NanCategorical_V1_0_1(values=[]),
            table=table,
            values=values,
        )

        collar_attributes.append(attribute)

print(collar_attributes)

Hole Type
Random Number
[CategoryAttribute_V1_1_0(table=LookupTable_V1_0_1(data='f5df61fa8d76d7ca691033f446972d880c4b32a4336ac20a716a8b7f8a1f9867', length=2, keys_data_type='int64', values_data_type='string'), values=IntegerArray1_V1_0_1(data='54bdde0040415318a624a438ee1d4006825105d4102f669f2e35246c5d3f774d', length=55, width=1, data_type='int32'), name='Hole Type', key='2435fdea-9eaa-40dc-b027-efda465cc5b7', attribute_type='category', attribute_description=None, nan_description=NanCategorical_V1_0_1(values=[])), ContinuousAttribute_V1_1_0(name='Random Number', key='742f47dc-27e0-4fe7-aa0a-119b02110a03', attribute_type='scalar', attribute_description=None, nan_description=NanContinuous_V1_0_1(values=[]), values=FloatArray1_V1_0_1(data='2ef1c0382b06497d938c9ca5fd1e640c636ddda0b796cdede7a455dff23f2e43', length=55, width=1, data_type='float64'))]


### Distances

In [11]:
import pandas as pd
from evo_schemas.elements import FloatArray3_V1_0_1

# Create a distances dataframe and copy the required columns.
# NOTE: The downhole collection object requires 3 columns: `final`, `target` and `current`,
# but in this data set there is only 1 depth value.
# To solve this problem we duplicate the depth value across the other 2 columns.
distances_df = pd.DataFrame()
distances_df["final"] = input_df["maxdepth"]
distances_df["target"] = distances_df.loc[:, "final"]
distances_df["current"] = distances_df.loc[:, "final"]

# Create a `FloatArray3_V1_0_1` element for the distances.
distances_component = FloatArray3_V1_0_1.from_dict(data_client.save_dataframe(distances_df))

### Survey

In [12]:
# Define input and output file paths.
input_file_path = f"{input_path}/Wolfpass_survey.csv"

# Load the survey file.
df = pd.read_csv(input_file_path)
df = df.sort_values([object_hole_id]).reset_index(drop=True)
df.head()

,BHID,AT,DIP,BRG,Random Numeric
0,WP001,0.000000,50.0,90.5,0.724666
1,WP001,31.000000,50.0,90.5,0.740499
2,WP001,198.050003,50.0,90.5,0.751197
3,WP002,0.000000,45.0,271.5,0.769103
4,WP002,31.000000,45.0,271.5,0.575993


### Path

In [13]:
# Remove hole IDs from the survey file that don't appear in the hole ID table.
# If you leave these rogue hole IDs in your survey table Leapfrog won't be able to import your geoscience object.
df.drop(df[~df[object_hole_id].isin(list(hole_id_table_df["value"]))].index, inplace=True)
df.reset_index(drop=True)
df.head()

,BHID,AT,DIP,BRG,Random Numeric
0,WP001,0.000000,50.0,90.5,0.724666
1,WP001,31.000000,50.0,90.5,0.740499
2,WP001,198.050003,50.0,90.5,0.751197
3,WP002,0.000000,45.0,271.5,0.769103
4,WP002,31.000000,45.0,271.5,0.575993


In [14]:
# Create a dataframe and copy the required survey column.
survey_values_df = pd.DataFrame()
survey_values_df["data"] = df[object_hole_id]

In [15]:
import pandas as pd
from evo_schemas.components import ContinuousAttribute_V1_1_0, NanContinuous_V1_0_1
from evo_schemas.elements import FloatArray1_V1_0_1

# Load the survey attributes

# Construct a FloatArray1 component for each attribute column in the survey table.
survey_attributes = []

for heading_name, heading_key in survey_attributes_ref.items():
    values_df = pd.DataFrame()
    values_df = df.loc[:, [heading_name]].copy().astype(float).reset_index(drop=True)
    values = FloatArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))

    attribute = ContinuousAttribute_V1_1_0(
        name=heading_name, key=heading_key, nan_description=NanContinuous_V1_0_1(values=[]), values=values
    )

    survey_attributes.append(attribute)

print(survey_attributes)

[ContinuousAttribute_V1_1_0(name='Random Numeric', key='731be77e-5586-450a-8950-1e6e97b02706', attribute_type='scalar', attribute_description=None, nan_description=NanContinuous_V1_0_1(values=[]), values=FloatArray1_V1_0_1(data='987ebf0333a015558e65f4690ea68d277db1167fc62d78243348bb709a9da543', length=423, width=1, data_type='float64'))]


In [16]:
# Rename the column names to what is expected in the geoscience object and drop the hold ID column.
df = df.rename(columns={"DIP": "dip", "AT": "distance", "BRG": "azimuth"})
# Drop the hole ID column and all columns listed in survey_attributes
cols_to_drop = [object_hole_id] + list(survey_attributes_ref.keys())
df = df.drop(columns=cols_to_drop)
df.head()

,distance,dip,azimuth
0,0.000000,50.0,90.5
1,31.000000,50.0,90.5
2,198.050003,50.0,90.5
3,0.000000,45.0,271.5
4,31.000000,45.0,271.5


In [17]:
# Swap the column order to match what the geoscience object
# requires (distance/azimuth/dip) and save the result in a new dataframe.
# NOTE: If your data set already has the correct ordering you can skip this step.
path_df = df.iloc[:, [0, 2, 1]]
path_df.head()

,distance,azimuth,dip
0,0.000000,90.5,50.0
1,31.000000,90.5,50.0
2,198.050003,90.5,50.0
3,0.000000,271.5,45.0
4,31.000000,271.5,45.0


In [18]:
from evo_schemas.objects import DownholeCollection_V1_3_0_Location_Path

# Create a `DownholeCollection_V1_3_0_Location_Path` object to handle the *path* component.
path_component = DownholeCollection_V1_3_0_Location_Path.from_dict(data_client.save_dataframe(path_df))
path_component.attributes = survey_attributes

### Holes

In [19]:
# Using the hole ID table from earlier, create a mapping between
# the hole IDs defined in the collar table and the ones found in the survey table.
holes_df = create_hole_id_mapping(hole_id_table=hole_id_table_df, value_list=survey_values_df)
holes_df.head()

,hole_index,offset,count
0,1,0,3
1,2,3,3
2,3,6,3
3,4,9,3
4,5,12,3


In [20]:
from evo_schemas.objects import DownholeCollection_V1_3_0_Location_Holes

# Create a `DownholeCollection_V1_3_0_Location_Holes` object to handle the *holes* component.
holes_component = DownholeCollection_V1_3_0_Location_Holes.from_dict(data_client.save_dataframe(holes_df))

In [21]:
from evo_schemas.objects import DownholeCollection_V1_3_0_Location

# Create a `DownholeCollection_V1_3_0_Location` object which combines all
# of the components created so far in this section.
location_component = DownholeCollection_V1_3_0_Location(
    distances=distances_component,
    holes=holes_component,
    hole_id=hole_id_component,
    path=path_component,
    coordinates=coordinates_component,
    attributes=collar_attributes,
)

### Distance table

In [22]:
# Enter the name of the input csv file (without the file extension) in the `collection_name` variable.
# The same name will be applied to this collection in the new geoscience object, but this is configurable.
collection_name = "Wolfpass_depths"
input_file_path = f"{input_path}/{collection_name}.csv"

# Load the assay file and find the length.
df = pd.read_csv(input_file_path)
orig_count = len(df)
df.head()

,BHID,DEPTH,ATTR
0,WP001,50.0,12.3
1,WP002,51.0,12.4
2,WP002,52.0,12.5
3,WP002,53.0,12.6
4,WP004,54.0,12.7


In [23]:
# Create an **depth_bhid_values** dataframe and copy the hole IDs.
depth_bhid_values_df = pd.DataFrame()
depth_bhid_values_df["data"] = df[object_hole_id]
depth_bhid_values_df.sort_values(by="data", inplace=True)
depth_bhid_values_df.reset_index(drop=True, inplace=True)
depth_bhid_values_df.head()

,data
0,WP001
1,WP001
2,WP001
3,WP002
4,WP002


#### Depths

In [24]:
# Create a dataframe and copy the *DEPTH* column to it.
depth_interval_table_df = df.loc[:, ["DEPTH"]].copy().reset_index(drop=True)
depth_interval_table_df.columns = ["data"]
depth_interval_table_df.head()

,data
0,50.0
1,51.0
2,52.0
3,53.0
4,54.0


In [25]:
from evo_schemas.elements import FloatArray1_V1_0_1

# Create a component that represents the depths.
depths = FloatArray1_V1_0_1.from_dict(data_client.save_dataframe(depth_interval_table_df))

#### Holes

In [26]:
# Create a mapping between the hole IDs in the depth table and the hole IDs from the collar table.
depth_holes_df = create_hole_id_mapping(hole_id_table=hole_id_table_df, value_list=depth_bhid_values_df)
depth_holes_df.head()

,hole_index,offset,count
0,1,0,3
1,2,3,6
2,3,0,0
3,4,9,2
4,5,0,0


In [27]:
from evo_schemas.objects import DownholeCollection_V1_3_0_Location_Holes

# Create a component that represents the depth holes.
location_holes_component = DownholeCollection_V1_3_0_Location_Holes.from_dict(
    data_client.save_dataframe(depth_holes_df)
)

#### Depth Attributes

In [28]:
import pandas as pd
from evo_schemas.components import ContinuousAttribute_V1_1_0, NanContinuous_V1_0_1
from evo_schemas.elements import FloatArray1_V1_0_1

# Construct a FloatArray1 component for each column in the assay table.
attributes = []

for heading_name, heading_key in depth_attribute_ref["Wolfpass_depths"].items():
    values_df = pd.DataFrame()
    values_df = df.loc[:, [heading_name]].copy().astype(float).reset_index(drop=True)
    values = FloatArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))

    attribute = ContinuousAttribute_V1_1_0(
        name=heading_name, key=heading_key, nan_description=NanContinuous_V1_0_1(values=[]), values=values
    )

    attributes.append(attribute)

In [29]:
from evo_schemas.components import DistanceTable_V1_2_0_Distance
from evo_schemas.objects import DownholeCollection_V1_3_0_Collections_DistanceTable

# Create the `DownholeCollection_V1_3_0_Collections_DistanceTable` object by combining the *holes* and *distance* components.
collection = DownholeCollection_V1_3_0_Collections_DistanceTable(
    name=collection_name,
    holes=location_holes_component,
    distance=DistanceTable_V1_2_0_Distance(values=depths, attributes=attributes),
)

collections.append(collection)

### Assay

In [30]:
# Enter the name of the input csv file (without the file extension) in the `collection_name` variable.
# The same name will be applied to this collection in the new geoscience object, but this is configurable.
collection_name = "Wolfpass_WP_assay"
input_file_path = f"{input_path}/{collection_name}.csv"

# Load the assay file and find the length.
df = pd.read_csv(input_file_path)
orig_count = len(df)
df.head()

,BHID,FROM,TO,CU_pct,AU_gpt,DENSITY
0,WP001,0.0,2.0,0.86,1.75,NS
1,WP001,2.0,4.0,0.83,1.73,NS
2,WP001,4.0,6.0,0.84,6.0,NS
3,WP001,6.0,8.0,0.83,2.56,2.32
4,WP001,8.0,10.0,0.97,1.53,2.98


In [31]:
# Remove hole IDs from the survey file that don't appear in the hole ID table.
# If you leave these rogue hole IDs in your survey table Leapfrog won't be able to import your geoscience object.
df.drop(df[~df[object_hole_id].isin(list(hole_id_table_df["value"]))].index, inplace=True)
df.reset_index(drop=True)
new_count = len(df)
print(f"Num IDs removed: {orig_count - new_count}")

Num IDs removed: 0


In [32]:
# Some rows in the example assay table contain invalid characters,
# eg. the AU_gpt column contains the string *NS* instead of a floating point number.
# In this example we replace the *NS* string with a value of *-999.0*.
# The table also contains some *less than* symbols (ie. '<') which also need to be removed.
df = df.replace(to_replace="NS", value=-999.0)
df = df.loc[df["AU_gpt"].str[:1] != "<"]

In [33]:
# We need to create a look-up table but the hole IDs must be grouped and ordered, so we sort the dataframe accordingly.
# We also count the number of rows in the final dataframe.
df.sort_values(by=[object_hole_id], inplace=True)
num_rows = len(df.index)

In [34]:
# Create an **assay_bhid_values** dataframe and copy the hole IDs.
assay_bhid_values_df = pd.DataFrame()
assay_bhid_values_df["data"] = df[object_hole_id]
assay_bhid_values_df.head()

,data
0,WP001
1,WP001
2,WP001
3,WP001
4,WP001


### Intervals

In [35]:
# Create a dataframe and copy the *FROM* and *TO* columns to it.
assay_interval_table_df = df.loc[:, ["FROM", "TO"]].copy().reset_index(drop=True)
assay_interval_table_df.columns = ["data1", "data2"]
assay_interval_table_df.head()

,data1,data2
0,0.0,2.0
1,2.0,4.0
2,4.0,6.0
3,6.0,8.0
4,8.0,10.0


In [36]:
from evo_schemas.elements import FloatArray2_V1_0_1

# Create a component that represents the intervals.
start_and_end = FloatArray2_V1_0_1.from_dict(data_client.save_dataframe(assay_interval_table_df))

### Holes

In [37]:
# Create a mapping between the hole IDs in the assay table and the hole IDs from the collar table.
assay_holes_df = create_hole_id_mapping(hole_id_table=hole_id_table_df, value_list=assay_bhid_values_df)

In [38]:
from evo_schemas.objects import DownholeCollection_V1_3_0_Location_Holes

# Create a component that represents the assay holes.
location_holes_component = DownholeCollection_V1_3_0_Location_Holes.from_dict(
    data_client.save_dataframe(assay_holes_df)
)

### Collection Attributes

In [39]:
import pandas as pd
from evo_schemas.components import ContinuousAttribute_V1_1_0, NanContinuous_V1_0_1
from evo_schemas.elements import FloatArray1_V1_0_1

# Construct a FloatArray1 component for each column in the assay table.
attributes = []

for heading_name, heading_key in object_attributes_ref["Wolfpass_WP_assay"].items():
    values_df = pd.DataFrame()
    values_df = df.loc[:, [heading_name]].copy().astype(float).reset_index(drop=True)
    values = FloatArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))

    attribute = ContinuousAttribute_V1_1_0(
        name=heading_name, key=heading_key, nan_description=NanContinuous_V1_0_1(values=[]), values=values
    )

    attributes.append(attribute)

In [40]:
from evo_schemas.components import Intervals_V1_0_1, IntervalTable_V1_2_0_FromTo
from evo_schemas.objects import DownholeCollection_V1_3_0_Collections_IntervalTable

# Create the `DownholeCollection_V1_3_0_Collections_IntervalTable` object by combining the *holes* and *from_to* components.
from_to_component = IntervalTable_V1_2_0_FromTo(
    intervals=Intervals_V1_0_1(start_and_end=start_and_end),
    attributes=attributes,
)

collection = DownholeCollection_V1_3_0_Collections_IntervalTable(
    name=collection_name, from_to=from_to_component, holes=location_holes_component
)

collections.append(collection)

### Lithology

In [41]:
# Enter the name of the input csv file (without the file extension) in the `collection_name` variable.
# The same name will be applied to this collection in the new geoscience object, but this is configurable.
collection_name = "Wolfpass_WP_lith"
input_file_path = f"{input_path}/{collection_name}.csv"

# Load the lithology file and find the length.
df = pd.read_csv(input_file_path)
orig_count = len(df)
df.head()

,BHID,FROM,TO,ROCK,grouped_lith,Split_Dykes
0,WP001,0.0,0.10,SAPR,Recent,Recent
1,WP001,0.1,0.80,SGNCRLSS,NaN,NaN
2,WP001,0.8,0.90,SAPR,Recent,Recent
3,WP001,0.9,1.80,SGNCRLSS,NaN,NaN
4,WP001,1.8,4.05,SAPR,Recent,Recent


In [42]:
# Remove hole IDs from the survey file that don't appear in the hole ID table.
# If you leave these rogue hole IDs in your survey table Leapfrog won't be able to import your geoscience object.
df.drop(df[~df[object_hole_id].isin(list(hole_id_table_df["value"]))].index, inplace=True)
df.reset_index(drop=True)
new_count = len(df)
print(f"Num IDs removed: {orig_count - new_count}")

Num IDs removed: 0


In [43]:
# We need to create a look-up table but the hole IDs must be grouped and ordered, so we sort the dataframe accordingly.
# We also count the number of rows in the final dataframe.
df.sort_values(by=[object_hole_id], inplace=True)
num_rows = len(df.index)

In [44]:
# Create a **lith_bhid_values** dataframe and copy the hole IDs.
lith_bhid_values_df = pd.DataFrame()
lith_bhid_values_df["data"] = df[object_hole_id]
lith_bhid_values_df.head()

,data
0,WP001
1,WP001
2,WP001
3,WP001
4,WP001


### Intervals

In [45]:
# Create a dataframe and copy the `FROM` and `TO` columns to it.
lith_interval_table_df = pd.DataFrame()
lith_interval_table_df = df.loc[:, ["FROM", "TO"]].copy().reset_index(drop=True)
lith_interval_table_df.head()

,FROM,TO
0,0.0,0.10
1,0.1,0.80
2,0.8,0.90
3,0.9,1.80
4,1.8,4.05


In [46]:
from evo_schemas.elements import FloatArray2_V1_0_1

# Create a component that presents the intervals.
start_and_end = FloatArray2_V1_0_1.from_dict(data_client.save_dataframe(lith_interval_table_df))

### Holes

In [47]:
# Create a mapping between the hole IDs in the lithology table and the hole IDs from the collar table.
lith_holes_df = create_hole_id_mapping(hole_id_table=hole_id_table_df, value_list=lith_bhid_values_df)

In [48]:
from evo_schemas.objects import DownholeCollection_V1_3_0_Location_Holes

# Create a component that represents the lithology holes.
location_holes_component = DownholeCollection_V1_3_0_Location_Holes.from_dict(data_client.save_dataframe(lith_holes_df))

### Collection Attributes

In [49]:
import pandas as pd
from evo_schemas.components import CategoryAttribute_V1_1_0, NanCategorical_V1_0_1
from evo_schemas.elements import IntegerArray1_V1_0_1, LookupTable_V1_0_1

# Construct a FloatArray1 component for each column in the assay table.
attributes = []

for heading_name, heading_key in object_attributes_ref["Wolfpass_WP_lith"].items():
    lith_df = pd.DataFrame()
    lith_df["data"] = df[heading_name]
    lith_df = lith_df.astype(str).reset_index(drop=True)
    table_df, values_df = create_category_lookup_and_values(lith_df)

    table = LookupTable_V1_0_1.from_dict(data_client.save_dataframe(table_df))
    values = IntegerArray1_V1_0_1.from_dict(data_client.save_dataframe(values_df))

    attribute = CategoryAttribute_V1_1_0(
        name=heading_name, key=heading_key, nan_description=NanCategorical_V1_0_1(values=[]), table=table, values=values
    )
    attributes.append(attribute)

In [50]:
from evo_schemas.components import Intervals_V1_0_1, IntervalTable_V1_2_0_FromTo
from evo_schemas.objects import DownholeCollection_V1_3_0_Collections_IntervalTable

# Create the `DownholeCollection_V1_3_0_Collections_IntervalTable` object by combining the *holes* and *from_to* components.
from_to_component = IntervalTable_V1_2_0_FromTo(
    intervals=Intervals_V1_0_1(
        start_and_end=start_and_end,
    ),
    attributes=attributes,
)

collection = DownholeCollection_V1_3_0_Collections_IntervalTable(
    name=collection_name, from_to=from_to_component, holes=location_holes_component
)

collections.append(collection)

### Assemble the downhole collection object

In [51]:
from evo_schemas.objects import DownholeCollection_V1_3_0

from evo.notebooks import FeedbackWidget

# Lastly, assemble the complete geoscience object by combining all previously defined components.
# - The name and UUID are used to identify the object.
# - The UUID is set to None because this is a new object. A new UUID will be assigned by the Evo service.
# - The bounding box defines the spatial extent of the object.
# - The tags provide metadata about the object.
# - The coordinate reference system defines the spatial reference for the object.
# - The locations component contains the coordinates and attributes.
downhole_collection = DownholeCollection_V1_3_0(
    name=object_name,
    uuid=None,
    bounding_box=bounding_box,
    tags=object_tags,
    coordinate_reference_system=coordinate_reference_system,
    location=location_component,
    collections=collections,
)

await data_client.upload_referenced_data(downhole_collection.as_dict(), FeedbackWidget("Uploading data"))
new_downhole_collection_metadata = await object_client.create_geoscience_object(
    full_obj_path, downhole_collection.as_dict()
)

### View the object in the Evo portal

In [52]:
build_portal_url(new_downhole_collection_metadata)

### Display the object in 3D

In [55]:
from evo.widgets import EvoObjectViewer, download_tileset_bundle

bundle = await download_tileset_bundle(
    manager,
    new_downhole_collection_metadata.id,
    name=getattr(new_downhole_collection_metadata, "name", object_name),
)
viewer = EvoObjectViewer()
viewer.add_bundle(bundle)
display(viewer)

Success! You now have a new geoscience object in Evo containing your downhole-collection data.

## Summary

In this example, we've completed the following:
* Analysed the collar and survey tables and constructed the elements and components required for locations.
* Analysed the data columns and constructed the elements and components required for attributes.
* Converted the input location and attribute data into Parquet format and saved it to the local cache.
* Combined all of the elements, components and data references into the downhole-collection schema format.
* Uploaded the Parquet files and the newly assembled object in JSON format to Evo.
* Displayed the new object in a 3D scene.